# OpenClaw Lab 1: Overall Harness

Before running the lab, let's first say what OpenClaw is.

OpenClaw is a platform for running personal AI agents. In a full setup, an OpenClaw agent can connect to chat apps, local devices, tools, automations, and multiple workspaces. It can act as a personal assistant, a service-side agent, a channel bot, or a local agent runtime that you control.

For now, just think of **OpenClaw** as a powerful workbench for running AI agents. OpenClaw can do much more than this lab will cover, but we will begin with the part that matters most for understanding an agent turn: connecting a language model to files, command-line tools, session history, and memory notes.

At first, this may feel like a black box. You give the agent a task, and it produces an answer. This lab opens that box step by step.

In this lab, you will play the role of a TA preparing course lab materials for an AI Agent and LLM Serving course. The lab workspace contains a course brief, FAQ, rubric, release checklist, memory note, and one sample student answer. Your job is to ask OpenClaw to inspect those materials and help decide whether the lab is ready for students.

We will start with the simplest case: asking a model to reply to one prompt. Then we will slowly add more pieces: course files, workspace inspection, CLI tools, session history, and memory.

Do not worry about the word **harness** yet. First run the cells, read the outputs, and write down what changes from one step to the next. We will name the system pieces after you have seen them work.

## Section 1: Run OpenClaw On Course Lab Materials

We will build your understanding of OpenClaw one layer at a time.

First, you will connect OpenClaw to a model. Then you will ask it to run a small agent task. After that, you will inspect the course files yourself, ask the agent to use those files, ask it to use CLI tools, compare session behavior, read a memory note, and finally combine everything into a short course-assistant report.

By the end of this section, you should have seen OpenClaw:

1. call a configured model,
2. run a simple agent task,
3. answer from course files,
4. use CLI tools to inspect the workspace,
5. continue a session,
6. read a durable memory note,
7. combine those pieces into a short lab release report.

For each step, focus on two questions: what did you ask OpenClaw to do, and what evidence in the output shows how it did it?



### 1. Choose Your Model Provider

**What you will do:** choose how OpenClaw will reach a language model.

OpenClaw is not a model by itself. It needs a **Model Provider**: a backend that OpenClaw can send prompts to and receive model outputs from. For this lab, you only need one working provider path.

Here you are not choosing the agent runtime yet. You are choosing how OpenClaw reaches a model. Later, `agent --local` will use the configured model to run a full agent task.

**Choose one provider path before continuing.**

- DeepSeek is a hosted API provider. Use it only if you have your own API key and accept any billing.
- Ollama is a local model provider. It can run inside Colab or on your own machine, but GPU is recommended for later agent tasks.
- If your class uses another provider, vLLM, SGLang, or an OpenAI-compatible endpoint, you can adapt the notebook to that backend.

OpenClaw often writes models as `provider/model`, such as `deepseek/deepseek-v4-flash` or `ollama/qwen2.5-coder:7b`. That is the model reference syntax. The main idea for now is simpler: OpenClaw needs a provider before it can call a model.

**What to look for:** after setup, OpenClaw should list a model and the later Hello World call should return a short answer. If model setup fails, fix that before running the agent tasks.


In [ ]:
import getpass
import os
import pathlib
import shlex
import shutil
import subprocess
from typing import Iterable
REPO_URL = "https://github.com/SleepyLGod/openclaw-teaching.git"
REPO_DIR = pathlib.Path("/content/openclaw-teaching")
LAB_DIR = REPO_DIR / "labs" / "overall-harness"
WORKSPACE_SRC = LAB_DIR / "fundamentals-workspace"
OPENCLAW_WORKSPACE = pathlib.Path.home() / ".openclaw" / "workspace"
# Change this if you use another configured API provider/model.
MODEL_REF = "deepseek/deepseek-v4-flash"
# Default for Colab T4/local GPU: better for workspace + CLI-style agent tasks.
# If your runtime is CPU-only or too small, change this to "llama3.2:3b"
# or another model exposed through your Ollama provider.
OLLAMA_MODEL_NAME = "qwen2.5-coder:7b"
OLLAMA_MODEL_REF = f"ollama/{OLLAMA_MODEL_NAME}"
def section(title: str) -> None:
    line = "=" * len(title)
    print(f"\n{line}\n{title}\n{line}")
def run(cmd: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    print(f"\n$ {cmd}")
    proc = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")
    return proc
def quote(text: str) -> str:
    return shlex.quote(text)
def show_file(path: pathlib.Path) -> None:
    print(f"\n--- {path} ---")
    print(path.read_text(encoding="utf-8"))
def show_files(paths: Iterable[pathlib.Path]) -> None:
    for path in paths:
        if path.is_file():
            show_file(path)


### 2. Set Up OpenClaw

**What you will do:** clone the teaching repo, install OpenClaw in this runtime, and copy the course lab materials into OpenClaw's workspace.

The workspace is the folder OpenClaw will use for this lab. It contains the files the agent will later inspect: `COURSE_BRIEF.md`, `FAQ.md`, `RUBRIC.md`, `LAB_RELEASE_CHECKLIST.md`, `MEMORY.md`, and a sample student answer.

This cell is also your first reading checkpoint. The notebook prints the Markdown files so you can inspect them yourself before asking the agent to use them. That gives you ground truth: later, you can compare the agent's answer against the actual file contents.

**What to look for:** read the printed file contents. Notice what each file is for. When the agent answers later, check whether it names the right files and uses details that really appear in those files.



In [ ]:
section("2. Set Up OpenClaw")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
_ = run(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")

print("Installing OpenClaw. This may take a minute.")
_ = run("curl -fsSL https://openclaw.ai/install.sh | bash -s -- --no-onboard", check=False)

os.environ["PATH"] = ":".join([
    str(pathlib.Path.home() / ".local" / "bin"),
    str(pathlib.Path.home() / ".openclaw" / "bin"),
    os.environ.get("PATH", ""),
])

_ = run("command -v openclaw || true", check=False)
_ = run("openclaw --version", check=False)
_ = run("openclaw doctor", check=False)

OPENCLAW_WORKSPACE.mkdir(parents=True, exist_ok=True)
for src in WORKSPACE_SRC.rglob("*"):
    if src.is_file():
        target = OPENCLAW_WORKSPACE / src.relative_to(WORKSPACE_SRC)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, target)

print("OpenClaw workspace target:", OPENCLAW_WORKSPACE)
print("\nCourse lab files copied into the workspace:")
workspace_files = sorted(
    path for path in OPENCLAW_WORKSPACE.rglob("*")
    if path.is_file() and path.name in {
        "COURSE_BRIEF.md",
        "LAB_RELEASE_CHECKLIST.md",
        "RUBRIC.md",
        "FAQ.md",
        "MEMORY.md",
        "sample_student_answer.md",
    }
)
show_files(workspace_files)


### 3. Configure A Model Provider

**What you will do:** run the DeepSeek provider cell, the Ollama provider cell, or another provider setup your instructor gives you.

**Choose one provider path before continuing.** You do not need to run both DeepSeek and Ollama. DeepSeek is usually more stable for a live classroom demo if you have a key. Ollama is useful for seeing a local-model path, but full agent tasks can be slow on CPU.

For your first run, pick one provider path and stay with it. If you want to compare providers, check the printed model reference before continuing.

**What to look for:** OpenClaw should show the configured provider and model. If you choose the Ollama cell, the notebook switches later tasks to the Ollama model automatically.


#### 3A. Optional API Route: DeepSeek

**What you will do:** connect OpenClaw to DeepSeek using your own API key.

If you are in Colab, the notebook first checks Colab Secrets for `DEEPSEEK_API_KEY`. If it is not there, it asks for hidden input. The key should not be printed in the notebook output.

**What to look for:** the model list should include DeepSeek models. If you do not have a key, skip this route and use a local or instructor-provided route instead.



In [ ]:
section("3A. Configure DeepSeek API route")
print("Model reference:", MODEL_REF)
print("This cell does not print your API key.")

if not os.environ.get("DEEPSEEK_API_KEY"):
    try:
        from google.colab import userdata
        secret = userdata.get("DEEPSEEK_API_KEY")
    except Exception:
        secret = None

    if secret:
        os.environ["DEEPSEEK_API_KEY"] = secret
        print("Loaded DEEPSEEK_API_KEY from Colab Secrets.")
    else:
        os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("Enter DEEPSEEK_API_KEY: ")
else:
    print("DEEPSEEK_API_KEY already exists in the environment.")

_ = run('openclaw onboard --non-interactive --mode local --auth-choice deepseek-api-key --deepseek-api-key "$DEEPSEEK_API_KEY" --skip-health --accept-risk', check=False)
_ = run("openclaw models list --all --provider deepseek", check=False)
_ = run(f"openclaw models set {shlex.quote(MODEL_REF)}", check=False)


#### 3B. Optional Local Provider: Ollama

**What you will do:** install Ollama in this Colab environment, start the local server, pull a model, and tell OpenClaw to use it.

The default model is `qwen2.5-coder:7b`. It is a better fit for workspace and CLI-style agent tasks than a very small model, and it usually fits a Colab T4 GPU. If your runtime has less memory, is CPU-only, or feels too slow, change `OLLAMA_MODEL_NAME` in the next code cell to a smaller model such as `llama3.2:3b`.

Local models are useful for learning, but they are not guaranteed to behave like stronger hosted models. A small local model may complete Hello World and still struggle when the full agent task includes files, tools, session history, and longer instructions. Treat that as an observation about local-model capacity, not as a failed lab.

**What to look for:** Ollama should start, the model should download, and OpenClaw should set the Ollama model for later cells.


In [ ]:
section("3B. Configure Ollama local route")
print("Ollama model:", OLLAMA_MODEL_NAME)
print("If you change OLLAMA_MODEL_NAME, rerun this cell before the OpenClaw tasks.")
os.environ["OLLAMA_API_KEY"] = "ollama-local"
_ = run("apt-get update", check=False)
_ = run("apt-get install -y zstd curl ca-certificates", check=False)
_ = run("curl -fsSL https://ollama.com/install.sh | sh", check=False)
_ = run("pgrep -x ollama >/dev/null || (nohup ollama serve > /tmp/ollama.log 2>&1 &)", check=False)
_ = run("sleep 5 && curl -s http://127.0.0.1:11434/api/tags || true", check=False)
_ = run(f"ollama pull {shlex.quote(OLLAMA_MODEL_NAME)}", check=False)
_ = run("openclaw models list --all --provider ollama", check=False)
_ = run(f"openclaw models set {shlex.quote(OLLAMA_MODEL_REF)}", check=False)
MODEL_REF = OLLAMA_MODEL_REF
print("Notebook model for later tasks:", MODEL_REF)


### 4. LLM Hello World

**What you will do:** send one short prompt directly to the selected model.

This is the smallest run in the notebook. OpenClaw sends exactly the prompt in the code cell to the selected Model Provider and prints the model's reply. No course files, tools, session history, or memory notes are needed for this step.

**What to look for:** the output should contain `OpenClaw is connected.` or something very close. This tells you the provider and model are connected. It does not yet show what makes an agent task different.


In [ ]:
section("4. LLM Hello World")
prompt = "Reply with exactly: OpenClaw is connected."
print("Command path: openclaw infer model run")
print("Model reference:", MODEL_REF)
_ = run(f"openclaw infer model run --local --model {shlex.quote(MODEL_REF)} --prompt {quote(prompt)} --json", check=False)


### 5. Agent Hello World

**What you will do:** run your first OpenClaw agent task.

The message is still short, but this time OpenClaw starts an agent turn with a session key and the workspace prepared earlier. You are now asking the agent workbench to handle a task, not only asking a model to complete one prompt.

**What to look for:** compare this output with the direct model call. You may see extra runtime messages, and the answer may sound more like a course assistant. At this point, the task is still simple; the next steps ask the agent to use files and tools.



In [ ]:
section("5. Agent Hello World")
message = "You are helping with an AI Agent course lab. In two sentences, say what kind of work you can help with in this workspace."
print("Command path: openclaw agent --local")
print("Session key: fundamentals-agent-hello")
_ = run(f"openclaw agent --local --session-key fundamentals-agent-hello --message {quote(message)}", check=False)


### 6. Ask About The Course Files

**What you will do:** ask the agent to read `COURSE_BRIEF.md` and `FAQ.md` before answering.

This is the first file-grounded task. The agent should use the lab workspace rather than answer from general knowledge.

**What to look for:** check whether the answer mentions concrete details from the course brief or FAQ. Because you already read the printed files in Step 2, you have a ground-truth reference. If the answer only gives generic statements about AI agents, note that as a behavior to investigate.



In [ ]:
section("6. Ask About The Course Files")
message = "Read COURSE_BRIEF.md and FAQ.md. Summarize what this lab is about and list two questions students may ask."
print("Files expected to matter: COURSE_BRIEF.md, FAQ.md")
_ = run(f"openclaw agent --local --session-key fundamentals-course-files --message {quote(message)}", check=False)


### 7. Use CLI Tools To Inspect The Workspace

**What you will do:** ask OpenClaw to inspect the lab workspace with CLI tools.

In this lab, a **tool** means an action OpenClaw can run for the agent. The model can request a tool call, but it does not directly own your filesystem and does not run shell commands by itself. OpenClaw decides which tools are available, runs the action, and returns results to the agent.

This is a real tool-use step, not fake agent code and not a calculator toy. The task is similar to what a TA might do before publishing lab materials: check that the required files exist and decide which file should be inspected first.

**What to look for:** the answer should name the files it found. If you see a tool-policy line, do not treat it as an error by default; it means OpenClaw is shaping which tools are available for this agent task.



In [ ]:
section("7. Use CLI Tools To Inspect The Workspace")
message = "Use CLI tools to inspect the lab workspace. Check whether COURSE_BRIEF.md, LAB_RELEASE_CHECKLIST.md, RUBRIC.md, FAQ.md, MEMORY.md, and submissions/sample_student_answer.md exist. Then report which files are present and which file you would inspect first before publishing the lab."
print("Expected action: use runtime-provided CLI/filesystem tools to inspect workspace files.")
_ = run(f"openclaw agent --local --session-key fundamentals-cli-tools --message {quote(message)}", check=False)


### 8. Continue The Same Session

**What you will do:** give the agent a temporary review preference, then ask it to review a sample student answer in the same session.

After that, you will ask a similar review question with a fresh session key.

**What to look for:** in the same session, the agent should use the review preference you just gave it. In the fresh session, that preference should not automatically carry over unless it comes from another source such as a workspace file or memory note.



In [ ]:
section("8. Continue The Same Session")
review_session = "fundamentals-review-session"
turn_1 = "For this session, when you review student answers, focus first on whether the answer cites observable OpenClaw behavior."
turn_2 = "Review submissions/sample_student_answer.md using the review preference I just gave you."
fresh_turn = "Review submissions/sample_student_answer.md. What review preference are you using?"

print("Same session key:", review_session)
_ = run(f"openclaw agent --local --session-key {review_session} --message {quote(turn_1)}", check=False)
_ = run(f"openclaw agent --local --session-key {review_session} --message {quote(turn_2)}", check=False)

print("\nFresh session key: fundamentals-review-fresh")
_ = run(f"openclaw agent --local --session-key fundamentals-review-fresh --message {quote(fresh_turn)}", check=False)


### 9. Use A Simple Memory Note

**What you will do:** ask the agent to read `MEMORY.md`, a small durable note in the workspace.

This is not a deep memory-system lab. For now, treat `MEMORY.md` as a simple example of a durable preference written down outside the immediate chat turn.

**What to look for:** the answer should explain the TA's feedback preference from `MEMORY.md`. Compare this with the temporary session preference from the previous step.



In [ ]:
section("9. Use A Simple Memory Note")
message = "Read MEMORY.md and tell me the durable TA preference for feedback style."
print("File expected to matter: MEMORY.md")
_ = run(f"openclaw agent --local --session-key fundamentals-memory-note --message {quote(message)}", check=False)


### 10. Complete A Small Course Assistant Task

**What you will do:** ask OpenClaw for a short release-readiness report.

Run this after the earlier cells if you want the full behavior. This task reuses the `fundamentals-review-session` session from Step 8, so the temporary review preference from that step can still matter. If you skipped Step 8 or changed the session key, the final report may not include that preference.

This final task combines the course files, FAQ, rubric, release checklist, memory note, current session preference, and workspace inspection into one useful answer.

**What to look for:** the exact wording will vary by model. A strong answer should mention which files or tools informed it. If the answer sounds plausible but gives no evidence, note that as a weakness.



In [ ]:
section("10. Complete A Small Course Assistant Task")
message = "You are preparing this OpenClaw lab for students. Use the course brief, FAQ, rubric, release checklist, memory note, and the current session preference. Produce a short release-readiness report with: 1. what the lab teaches, 2. what students must observe, 3. one likely student confusion, 4. one suggested improvement before publishing. Mention which files or tools informed your answer."
print("Session key: fundamentals-review-session")
print("Expected inputs: course files, memory note, prior review preference, and tool-visible workspace state.")
_ = run(f"openclaw agent --local --session-key fundamentals-review-session --message {quote(message)}", check=False)


## Section 2: What Was Around The Model?

We have already seen the behavior. Now we name the pieces.

In Section 1, the direct model call was small: one prompt went to one model and came back as one answer. The later OpenClaw runs felt different because the model was no longer working alone. It had a workspace, course files, tools, action rules, session history, and a memory note around it.

People use the word **harness** for this kind of setup, but the word is used at different levels. This section explains the idea first, then maps it back to what you just ran in OpenClaw.

A short version: an agent run is model + context + tools + state + policy + loop + evidence.

You do not need to run new code here. Read this as a guided explanation of the runs you just performed.


### 1. From The Demo To The Concept

Start with a concrete comparison.

If you ask for a course release report through a direct model call, the model only sees the prompt you typed. It does not automatically know your course files, your checklist, your session preference, or your memory note.

If you ask for the same report through an OpenClaw agent run, OpenClaw can prepare a richer work setting first.

```text
Direct model call

one prompt -> selected model -> answer


OpenClaw agent run

task
  -> workspace files
  -> tools and action rules
  -> session history
  -> memory note
  -> selected model
  -> answer with evidence
```

The model did not become a different kind of intelligence. The work setting around the model changed.

This matters because agent problems are often setup problems. The task may be vague. A file may be missing. A tool may be unavailable. The session may contain old instructions. The answer may not cite evidence. Harness engineering is the practice of finding and improving those setup pieces.


### 2. Reading The Harness Picture

The word **harness** is broad. Martin Fowler's article uses this picture to show that the meaning changes depending on where you draw the boundary.

![Fowler bounded-context harness diagram](https://martinfowler.com/articles/harness-engineering/harness-bounded-contexts.png)

*Source: [Martin Fowler, Harness Engineering](https://martinfowler.com/articles/harness-engineering.html).*

Read the picture from the inside out:

1. **Inner model.** This is the language model itself. It predicts and writes text.
2. **OpenClaw-built harness.** This is what OpenClaw adds around the model: runtime, prompt assembly, workspace handling, tool execution, action rules, and session handling.
3. **Our course harness.** This is what we add around OpenClaw for this class: course files, rubrics, release checklists, memory notes, review questions, and future evals.

So in this lab, “overall harness” does not mean one magic component. It means the whole prepared work setting around the model. Some of it is built by OpenClaw. Some of it is built by us for the course task.


### 3. A Definition We Will Use

There is no single universal definition of **harness**. We will use two sources to set the boundary.

Martin Fowler notes that one broad usage is: **“everything in an AI agent except the model itself.”** That is useful, but very broad. It includes many things: instructions, files, tools, tests, logs, review steps, and feedback loops.

OpenClaw uses the word more narrowly in its runtime documentation: **“A harness is the implementation that provides an agent runtime.”** In other words, OpenClaw also has a source-code meaning for harness. We will study that implementation later.

For this lab, use the broad teaching meaning:

> A harness is the setup around the model that helps it do a task reliably: files, instructions, tools, action rules, session history, memory notes, checks, and feedback loops.

Keep these five things separate:

| Do not confuse... | Meaning in this lab |
| --- | --- |
| Model | DeepSeek, an Ollama model, or another LLM that generates text |
| Model Provider | The route OpenClaw uses to reach the model, such as DeepSeek API or Ollama |
| Prompt | One piece of input sent to the model |
| Agent run | The full OpenClaw task turn started by `openclaw agent --local` |
| Harness | The files, tools, rules, session, memory, and checks around that run |

The useful habit is simple: when an agent gives a weak answer, do not only ask whether the model is strong enough. Ask what part of the work setting was weak.


### 4. Before The Agent Acts: Guides

Before an agent works, we can give it guides. In Fowler's article, **Guides** is the explicit term: guides are feedforward controls that steer the agent before it acts.

In plain language, a guide is anything the agent can use before doing the task: instructions, reference docs, examples, tool descriptions, or project rules.

![Fowler harness overview diagram](https://martinfowler.com/articles/harness-engineering/harness-overview.png)

*Source: [Martin Fowler, Harness Engineering](https://martinfowler.com/articles/harness-engineering.html). The left side shows guides before work starts. The right side shows feedback sensors after work happens.*

In Section 1, these were guides:

| Guide | How it helped |
| --- | --- |
| `COURSE_BRIEF.md` | Explained what the lab is about |
| `FAQ.md` | Gave likely student questions |
| `RUBRIC.md` | Described what a good answer should include |
| `LAB_RELEASE_CHECKLIST.md` | Gave publishing criteria |
| `MEMORY.md` | Stored a durable TA preference |
| Tool descriptions | Told the model what actions OpenClaw can run |
| The user task | Told the agent what to produce now |

This is why harness engineering is bigger than prompt engineering. A prompt is one guide. A real agent run often needs many guides, and we must decide which ones belong in the workspace, which ones belong in memory, which ones belong in a tool description, and which ones belong in the user task.


### 5. After The Agent Runs: What Can We Check?

After an agent run, we need evidence. We need to know what files were used, what tools were available, whether session history mattered, and whether the answer can be checked against the workspace.

Fowler's article calls these feedback controls **Sensors**. In this lab, read that word simply: a sensor is a check or observation point, and a signal is the evidence it leaves behind.

In Section 1, these were simple checks and signals:

| Sensor or check | Signal to read |
| --- | --- |
| Notebook prints workspace files | Can you compare the answer with the real course files? |
| CLI inspection result | Do the expected workspace files exist? |
| Tool-policy output | Which actions did OpenClaw allow or remove? |
| Same-session vs fresh-session comparison | Did session history affect the answer? |
| Final report evidence | Did the answer cite files or tools, or was it generic? |

Later chapters will add stronger checks: evals, guardrail checks, test cases, traces, review agents, and logs.

The main point is practical: an agent run should leave evidence. If you cannot tell what happened, you cannot improve the harness.


### 6. Two Ways To Check The Agent's Work

Some checks are easy for a program. Some checks require judgment.

Ask yourself: can a normal program check this, or does someone need to judge the meaning?

| Question | Best check | Lab example |
| --- | --- | --- |
| Does `FAQ.md` exist? | Program check | CLI inspection can list files |
| Did the notebook code parse? | Program check | `ast.parse` can validate Python cells |
| Did the answer cite observable OpenClaw behavior? | Judgment check | A human or review agent reads the answer |
| Is the explanation clear for students? | Judgment check | TA/professor review or model-as-reviewer |

Use program checks whenever they fit. They are fast and reliable. Use judgment checks when the question is about meaning, clarity, or pedagogy. Fowler calls these **computational** and **inferential** checks; the names matter less than choosing the right kind of check.

A strong harness usually combines both. For example, first check that the files exist; then review whether the answer used them well.


### 7. When The Output Is Weak, What Should We Change?

Harness engineering becomes useful when something goes wrong. Instead of saying “try again” or “use a bigger model,” inspect the setup.

Use these examples from the lab:

| If you see this output... | First question to ask | Smallest useful change |
| --- | --- | --- |
| The answer is generic | Did we give enough course-specific guidance? | Improve `COURSE_BRIEF.md`, the user task, or the rubric |
| The agent misses a required file | Is the file actually in the workspace? | Check setup cell output and file names |
| The agent cannot inspect files | Are the needed tools available? | Check tool output and action rules |
| A previous instruction leaks into a new run | Are we using the same `--session-key`? | Use a fresh session key or reset the session |
| The final report sounds good but cites no evidence | Did we ask for evidence? | Add “mention which files or tools informed your answer” |

This is the steering loop:

```text
run the agent
  -> inspect the result
  -> identify the weak part of the setup
  -> improve the guide, tool, rule, session, memory, or check
  -> run again
```

OpenAI's harness engineering post describes a similar shift: engineers increasingly spend time designing agent environments, stating intent clearly, and building feedback loops. In this lab, the loop is small, but the habit is the same.


### 8. What Kind Of Harness Are We Studying?

Fowler later separates harness goals into maintainability, architecture fitness, and behavior. Those categories are useful for coding agents, but this first OpenClaw lab does not need students to memorize them.

For now, we are studying the **overall shape** of a harness:

- guides before the agent acts,
- tools and rules while it acts,
- session and memory across turns,
- checks and evidence after it acts,
- a loop for improving the setup.

Later chapters will go deeper. Guardrails will focus on action rules. Eval will focus on checks, evidence, and review. Memory will focus on durable state. Skills will focus on reusable guidance. Source walkthroughs will show how OpenClaw implements these pieces.


### 9. OpenClaw Checkpoint

Use this table to diagnose the runs from Section 1. The goal is not to memorize a term. The goal is to know where to look first.

| If this step has a problem... | Inspect this first | Why |
| --- | --- | --- |
| LLM Hello World | Model Provider + Model | The direct call only proves the selected model can answer |
| Agent Hello World | Agent run setup | OpenClaw should start a full task turn |
| Ask About The Course Files | Workspace and task wording | The agent needs the right files and a clear request |
| Use CLI Tools To Inspect The Workspace | Tools and action rules | OpenClaw must expose and allow the needed actions |
| Tool-policy output is surprising | Action rules | Tool availability is being shaped by policy |
| Same-session behavior is surprising | Session key and history | The same key carries short-term task history |
| Fresh session still remembers old context | Session reset or hidden workspace/memory source | Something outside the immediate turn may be influencing the answer |
| Memory note is ignored | `MEMORY.md` and task wording | The agent may need a clearer instruction to read it |
| Final report has no evidence | Feedback requirement | Ask the agent to mention which files or tools informed the answer |

After this section, you should be able to say this in your own words:

> OpenClaw did more than call a model. It prepared files, tools, action rules, session history, memory notes, and evidence around the model. When an agent output is weak, check those pieces before you only blame the model.

Optional reading:

- Martin Fowler: https://martinfowler.com/articles/harness-engineering.html
- OpenAI: https://openai.com/index/harness-engineering/


## Section 3: Open The OpenClaw Source

You have now used OpenClaw as a black box. You connected a model, ran a direct model call, ran agent tasks, asked the agent to inspect files, compared sessions, read a memory note, and produced a small course-assistant report.

Now open the box. In this section, you will trace one `openclaw agent --local` run through the source code. The goal is not to memorize every file path. The goal is to see how the pieces from Section 1 and Section 2 are connected in a real implementation.

Guardrails and Eval appear here only as previews. This section shows where action rules and runtime evidence enter the overall harness. It does not teach Guardrails or Eval in full.

Source version used for this walkthrough:

- Repository: https://github.com/openclaw/openclaw
- Commit: `d4b4a658094e50ce964a479bd3d117b9067681af`
- Verified on: 2026-06-22
- Permalink base: https://github.com/openclaw/openclaw/blob/d4b4a658094e50ce964a479bd3d117b9067681af/

Source paths below are relative to `https://github.com/openclaw/openclaw`; if a path moves, search by function name. Snippets come from the pinned source commit above; `// ...` only hides unrelated real source lines.

Before reading the source, keep these words simple:

| Term | Plain meaning in this section |
| --- | --- |
| `attempt` | One try to finish an agent task. The user gives OpenClaw a task, and OpenClaw attempts to run it. |
| `lifecycle` | The run stages OpenClaw records, such as preparing, sending, resolving, and finishing. |
| `classification` | How the attempt ended, such as completed, failed, timed out, or blocked. |
| `bootstrap context` | Workspace files OpenClaw prepares before an agent run. |
| `provider stream` | The output channel between OpenClaw and the selected model provider. |
| `tool policy` | The rules that decide which tools are available, removed, or blocked. |
| `transcript` | Saved messages, tool results, and runtime records for a session. |


### 1. Trace One OpenClaw Agent Run

Start with the whole path. This is the source-level version of the agent task you ran in Section 1:

| Step | Source anchor | What happens |
| --- | --- | --- |
| Start an agent task | `openclaw agent --local` | The CLI receives the user task. |
| Build an attempt | `src/agents/agent-command.ts`, `runAgentAttempt` | OpenClaw bundles provider/model, session, workspace, current directory, config, and user message into one attempt. |
| Choose an executor | `selectAgentHarness(...)` | OpenClaw decides which runtime will execute the prepared attempt. |
| Run the default runtime | `createOpenClawAgentHarness()`, `runEmbeddedAttempt` | The built-in OpenClaw runtime prepares context, tools, action rules, session, and memory notes. |
| Execute the loop | `agentLoop(...)` | The model can answer, request tools, receive tool results, and continue. |
| Save evidence | transcript and diagnostics | OpenClaw leaves records you can inspect after the run. |

Keep three levels apart while reading:

- In this course, **harness engineering** means the work setup around the model: files, tools, action rules, session, memory, checks, and improvement loops.
- In OpenClaw documentation, the closest implementation idea is the **agent runtime**: the system that executes one prepared agent turn.
- In OpenClaw source code, `AgentHarness` is a TypeScript contract for one runtime executor.

These are connected, but they are not the same object. This section follows the source-level path and then maps it back to the broader course concept.


### 2. Model Provider Is Not The Runtime

In Section 1, you chose a Model Provider path such as DeepSeek or Ollama. That choice matters, but it is not the whole harness.

A provider answers the question: which backend serves the model?

A runtime answers a different question: who prepares and executes the agent turn?

OpenClaw resolves a model reference first:

```ts
export function resolveDefaultModelForAgent(
  params: {
    cfg: OpenClawConfig;
    agentId?: string;
    allowPluginNormalization?: boolean;
  } & ModelManifestNormalizationContext,
): ModelRef {
  const agentModelOverride = params.agentId
    ? resolveAgentEffectiveModelPrimary(params.cfg, params.agentId)
    : undefined;
  // ...
}
```

Source: `src/agents/model-selection.ts`

Inside the embedded attempt, OpenClaw registers a provider stream for the selected model:

```ts
const providerStreamFn = registerProviderStreamForModel({
  model: params.model,
  cfg: params.config,
  agentDir,
  workspaceDir: effectiveWorkspace,
});
```

Source: `src/agents/embedded-agent-runner/run/attempt.ts`

Provider plugins define concrete backend details. For example, DeepSeek builds a provider catalog from its manifest:

```ts
const DEEPSEEK_MANIFEST_PROVIDER = buildManifestModelProviderConfig({
  providerId: "deepseek",
  catalog: manifest.modelCatalog.providers.deepseek,
});

export const DEEPSEEK_BASE_URL = DEEPSEEK_MANIFEST_PROVIDER.baseUrl;
export const DEEPSEEK_MODEL_CATALOG: ModelDefinitionConfig[] = DEEPSEEK_MANIFEST_PROVIDER.models;
```

Source: `extensions/deepseek/models.ts`

Ollama resolves a local API base:

```ts
export function resolveOllamaApiBase(configuredBaseUrl?: string): string {
  if (!configuredBaseUrl) {
    return OLLAMA_DEFAULT_BASE_URL;
  }
  const trimmed = configuredBaseUrl.replace(/\/+$/, "");
  return trimmed.replace(/\/v1$/i, "");
}
```

Source: `extensions/ollama/src/provider-models.ts`

Changing DeepSeek to Ollama, vLLM, SGLang, or another OpenAI-compatible endpoint mostly changes the model-provider path. The OpenClaw agent runtime still prepares and runs the agent task.


### 3. From `agent --local` To An Agent Attempt

In Section 1, `openclaw agent --local` felt different from `openclaw infer model run`. The agent command starts a larger orchestration path.

Near the execution path, `agent-command.ts` passes provider/model, session, workspace, current directory, and the user request into one attempt:

```ts
return attemptExecutionRuntime.runAgentAttempt({
  providerOverride,
  modelOverride,
  modelFallbacksOverride: effectiveFallbacksOverride,
  originalProvider: provider,
  cfg,
  sessionEntry: attemptSessionEntry,
  sessionId,
  sessionKey,
  sessionAgentId,
  sessionFile: attemptSessionFile,
  workspaceDir,
  cwd,
  body,
  // ...
});
```

Source: `src/agents/agent-command.ts`

Read the names in plain language:

- `providerOverride` and `modelOverride` are the provider/model choices for this run if the command or config overrides the default.
- `sessionKey` is the user-facing session name, such as `fundamentals-review-session` in Section 1.
- `sessionFile` is where OpenClaw can store the session history for that key.
- `workspaceDir` is the agent workspace. In this lab, that is where the course files were copied.
- `cwd` is the current command directory used for command execution.
- `body` is the user task text.

The runtime barrel shows that `runAgentAttempt` is the attempt execution entry exported to callers:

```ts
// Runtime barrel for attempt execution. Kept separate so callers can import the
// light shared helpers without pulling the full command attempt graph.
export {
  // ...
  runAgentAttempt,
  sessionFileHasContent,
} from "./attempt-execution.js";
```

Source: `src/agents/command/attempt-execution.runtime.ts`

So `agent --local` does more than send `body` to a model. It packages one agent run: provider/model, session, workspace, current directory, config, and user task. Later sections follow those values deeper into the runtime.


### 4. The `AgentHarness` Contract

Here is the source-level meaning of `harness`. In OpenClaw core, an `AgentHarness` is the standard shape for something that can run an agent task.

Think of it like a contract. OpenClaw does not need to know every internal detail of a runtime, but it does need the runtime to answer two questions:

- Can you handle this task with the current provider/model?
- If yes, how do you run this task and return the result?

The source names for those two questions are `supports(...)` and `runAttempt(...)`:

```ts
export type AgentHarnessRunCapability = {
  id: string;
  label: string;
  pluginId?: string;
  contextEngineHostCapabilities?: readonly import("../../context-engine/types.js").ContextEngineHostCapability[];
  deliveryDefaults?: AgentHarnessDeliveryDefaults;
  supports(ctx: AgentHarnessSupportContext): AgentHarnessSupport;
  runAttempt(params: AgentHarnessAttemptParams): Promise<AgentHarnessAttemptResult>;
};
```

Source: `src/agents/harness/types.ts`

For this lab, read the last two lines first.

`supports(...)` is the yes/no gate. In our notebook, it answers a question like: can this runtime handle the current DeepSeek or Ollama agent task?

`runAttempt(...)` is the actual work. It runs one prepared agent task: prepare context, make tools available, apply action rules, manage session state, call the model loop, and return a result.

The other fields become easier to understand after you see a real runtime object in the next section. For now, just notice that a runtime has an internal name, a readable name, optional context-engine compatibility information, and the two important methods above.

OpenClaw then adds optional runtime abilities around the main run:

```ts
export type AgentHarness = AgentHarnessRunCapability &
  AgentHarnessSideQuestionCapability &
  AgentHarnessClassificationCapability &
  AgentHarnessCompactionCapability &
  AgentHarnessSessionLifecycleCapability;
```

Source: `src/agents/harness/types.ts`

You do not need to memorize these type names. Read them as optional helper abilities. A runtime may support small side questions, label how a run ended, compact an overlong session, or reset/dispose session resources. The main idea stays the same: `AgentHarness` is the source-level contract for a runtime executor.


### 5. Agent Runtime And Runtime Selection

An agent runtime is the code that executes one prepared agent turn. It receives the prepared attempt, runs the model/tool loop, and returns the result.

Start with the runtime used by this notebook. The DeepSeek and Ollama paths use OpenClaw's built-in runtime:

```ts
/** Creates the built-in harness backed by the embedded OpenClaw agent runner. */
export function createOpenClawAgentHarness(): AgentHarness {
  return {
    id: "openclaw",
    label: "OpenClaw embedded agent",
    contextEngineHostCapabilities: OPENCLAW_EMBEDDED_CONTEXT_ENGINE_HOST.capabilities,
    supports: () => ({ supported: true, priority: 0 }),
    runAttempt: runEmbeddedAttempt,
  };
}
```

Source: `src/agents/harness/builtin-openclaw.ts`

Read this object from top to bottom:

`id: "openclaw"` is the internal runtime name. `label: "OpenClaw embedded agent"` is the readable name that can appear in logs or status output. You can ignore `contextEngineHostCapabilities` in this lab; it only says this runtime can work with OpenClaw's context engine.

The important part is the last two lines. `supports: () => ({ supported: true, priority: 0 })` means the built-in runtime can accept the default path. `runAttempt: runEmbeddedAttempt` means the actual implementation continues in `runEmbeddedAttempt`.

Runtime selection answers one question: which executor will run this agent task? It is different from model selection. Model selection chooses a provider/model path such as DeepSeek or Ollama. Runtime selection chooses the code path that executes the prepared agent turn.

The public selection function is small:

```ts
export function selectAgentHarness(params: {
  provider: string;
  modelId?: string;
  config?: OpenClawConfig;
  agentId?: string;
  sessionKey?: string;
  agentHarnessId?: string;
  agentHarnessRuntimeOverride?: string;
}): AgentHarness {
  return selectAgentHarnessDecision(params).harness;
}
```

Source: `src/agents/harness/selection.ts`

Do not read every field in this snippet as a setting students should configure. The important point is simpler: OpenClaw has the provider/model, the session, and the runtime policy, then it chooses an `AgentHarness` to execute the turn.

Inside the decision path, OpenClaw compares registered plugin runtimes with the built-in OpenClaw runtime. `policy.runtime` is the runtime choice from config or policy. It is not the model provider.

```ts
const pluginHarnesses = listPluginAgentHarnesses();
const openClawHarness = createOpenClawAgentHarness();
const runtime = policy.runtime;
if (runtime === "openclaw") {
  return buildSelectionDecision({
    harness: openClawHarness,
    policy,
    selectedReason: "forced_openclaw",
    candidates: listHarnessCandidates(pluginHarnesses),
  });
}
// ...
```

Source: `src/agents/harness/selection.ts`

After selection, OpenClaw runs the chosen harness through lifecycle handling:

```ts
const runAttempt = () => runAgentHarnessLifecycleAttempt(harness, attemptParams);
if (harness.id === "openclaw") {
  return await runWithDiagnosticTraceContext(harnessTrace, runAttempt);
}
```

Source: `src/agents/harness/selection.ts`

So the runtime story for this notebook is: OpenClaw selects an executor, the selected executor is the built-in `openclaw` runtime, and the actual attempt is run by `runEmbeddedAttempt`. DeepSeek versus Ollama is still only the model-provider choice.


### 6. Direct Model Call Versus Agent Run

In Section 1, `LLM Hello World` used `openclaw infer model run`. That command directly asks the selected provider/model for text. It is useful for checking that the model can answer.

`openclaw agent --local` is a different kind of run. OpenClaw prepares a task environment first: files, instructions, tools, action rules, session history, and memory notes. Then it asks the model to work inside that prepared environment.

The embedded attempt code explicitly detects raw model runs:

```ts
const contextInjectionMode = resolveContextInjectionMode(params.config, sessionAgentId);
const isRawModelRun = params.modelRun === true || params.promptMode === "none";
// ...
const activeContextEngine = isRawModelRun ? undefined : params.contextEngine;
```

Source: `src/agents/embedded-agent-runner/run/attempt.ts`

In plain language, `isRawModelRun` means "this is just a model call." `contextInjectionMode` controls whether OpenClaw should inject prepared context into the run. For raw model calls, OpenClaw turns that off.

The same path keeps bootstrap context out of raw provider probes. Bootstrap context means files OpenClaw prepares before an agent run, such as root guidance or memory files.

```ts
const {
  bootstrapFiles: hookAdjustedBootstrapFiles,
  contextFiles: resolvedContextFiles,
  shouldRecordCompletedBootstrapTurn,
} = await resolveAttemptBootstrapContext({
  // modelRun is a provider probe, not an agent turn. Keep AGENTS/BOOTSTRAP
  // context out even when the gateway is exercising the embedded runtime.
  contextInjectionMode: isRawModelRun ? "never" : contextInjectionMode,
  bootstrapContextMode: params.bootstrapContextMode,
  bootstrapContextRunKind: params.bootstrapContextRunKind ?? "default",
  bootstrapMode,
  sessionFile: params.sessionFile,
  hasCompletedBootstrapTurn: hasCompletedBootstrapTurnForAttempt,
  resolveBootstrapContextForRun: async () => {
    const bootstrapFiles =
      preloadedBootstrapFiles ??
      (await resolveBootstrapFilesForRun({
        workspaceDir: resolvedWorkspace,
        config: params.config,
        // ...
      }));
    return {
      bootstrapFiles,
      contextFiles: buildBootstrapContextForFiles(bootstrapFiles, {
        config: params.config,
        agentId: sessionAgentId,
        warn: bootstrapWarn,
      }),
    };
  },
});
```

Source: `src/agents/embedded-agent-runner/run/attempt.ts`

The system prompt builder also treats raw runs differently. It still builds a base prompt for diagnostics and cache boundaries, but submits an empty provider system prompt for raw runs:

```ts
export function buildAttemptSystemPrompt(
  params: BuildAttemptSystemPromptParams,
): AttemptSystemPrompt {
  const baseSystemPrompt = buildEmbeddedSystemPrompt(params.embeddedSystemPrompt);
  const systemPrompt = params.isRawModelRun
    ? ""
    : params.transformProviderSystemPrompt({
        provider: params.providerTransform.provider,
        config: params.providerTransform.config,
        workspaceDir: params.providerTransform.workspaceDir,
        context: {
          ...params.providerTransform.context,
          systemPrompt: baseSystemPrompt,
        },
      });

  return {
    baseSystemPrompt,
    systemPrompt,
  };
}
```

Source: `src/agents/embedded-agent-runner/run/attempt-system-prompt.ts`

Here the split is concrete: `infer model run` is a narrow model call. `agent --local` enters the runtime path where context, tools, policy, session, and memory note can participate.

Do not overstate the contrast. A raw model can use file content if you paste that content into the prompt. The difference is who constructs the input and who owns tool actions, session, and evidence.


### 7. Workspace, Context, And System Prompt

In Section 1, the notebook printed the course files first. Then you asked the agent to use `COURSE_BRIEF.md`, `FAQ.md`, `RUBRIC.md`, and `MEMORY.md`.

There are two kinds of files to keep separate:

- Bootstrap/context files are guidance files that OpenClaw may prepare for the agent run. `MEMORY.md` is the visible example in this lab.
- Ordinary task files are work materials in the workspace. `COURSE_BRIEF.md`, `FAQ.md`, and `RUBRIC.md` belong here. The agent can use them through workspace and tool paths, but this section should not imply that every workspace file is automatically pasted into the prompt.

A task file being in the workspace does not guarantee that its full content is already in the model prompt. The agent may need to inspect it through tools, or OpenClaw may include selected bounded context depending on context mode.

| File or input type | Example | How it affects the run |
| --- | --- | --- |
| Guidance/bootstrap file | `AGENTS.md`, `SOUL.md`, `MEMORY.md` | May be prepared into system/project context depending on config and context mode. |
| Ordinary workspace task file | `COURSE_BRIEF.md`, `FAQ.md`, `RUBRIC.md` | The agent may inspect it through workspace/file/tool paths. Do not assume the full file is automatically pasted into the prompt. |
| Tool result | CLI output, file read result | Enters the agent loop as tool result messages. |
| Direct prompt text | User pastes content into the message | Becomes part of the user message. |

`bootstrap-files.ts` states the bootstrap job directly:

```ts
/**
 * Resolves workspace bootstrap files for agent runs and converts them into
 * bounded context files.
 */
```

Source: `src/agents/bootstrap-files.ts`

Workspace guidance files have known names. `MEMORY.md` is connected to the canonical root memory filename:

```ts
export const DEFAULT_AGENTS_FILENAME = "AGENTS.md";
export const DEFAULT_SOUL_FILENAME = "SOUL.md";
export const DEFAULT_TOOLS_FILENAME = "TOOLS.md";
export const DEFAULT_IDENTITY_FILENAME = "IDENTITY.md";
export const DEFAULT_USER_FILENAME = "USER.md";
export const DEFAULT_HEARTBEAT_FILENAME = "HEARTBEAT.md";
export const DEFAULT_BOOTSTRAP_FILENAME = "BOOTSTRAP.md";
export const DEFAULT_MEMORY_FILENAME = CANONICAL_ROOT_MEMORY_FILENAME;
```

Source: `src/agents/workspace.ts`

The system prompt code shows actual strings that can be assembled into the prompt. This is stronger evidence than only saying "context is added."

```ts
lines.push("The following project context files have been loaded:");
if (hasSoulFile) {
  lines.push("SOUL.md: persona/tone. Follow it unless higher-priority instructions override.");
}
if (hasMemoryFile) {
  lines.push(
    "MEMORY.md: durable user preferences and behavior guidance. Keep following it throughout the session unless higher-priority instructions override.",
  );
}
// ...
for (const file of params.files) {
  lines.push(`## ${file.path}`, "", sanitizeContextFileContentForPrompt(file.content), "");
}
```

Source: `src/agents/system-prompt.ts`

The system prompt also includes sections such as workspace files, safety, and runtime:

```ts
lines.push(
  // ...
  "## Workspace Files (injected)",
  "These user-editable files are loaded by OpenClaw and included below in Project Context.",
  "",
  // ...
);

lines.push(
  "## Runtime",
  buildRuntimeLine(runtimeInfo, runtimeChannel, runtimeCapabilities, params.defaultThinkLevel),
  ...(modelIdentityLine ? [modelIdentityLine] : []),
  // ...
);
```

Source: `src/agents/system-prompt.ts`

The model does not own the filesystem by itself. OpenClaw decides which guidance files become prompt context, and it provides workspace/tool paths for task files that the agent needs to inspect.


### 8. Tools And Action Rules

In Section 1, you asked the agent to use CLI tools to inspect the workspace. A concrete example was checking whether files such as `COURSE_BRIEF.md`, `FAQ.md`, `RUBRIC.md`, `LAB_RELEASE_CHECKLIST.md`, `MEMORY.md`, and `submissions/sample_student_answer.md` exist.

That is not a calculator toy. It is a real action: the agent needs some way to inspect the filesystem. The model can request an action, but OpenClaw decides which tools exist, which tools are allowed, how they run, and how tool results return to the run.

`agent-tools.ts` shows that tool construction receives the same kinds of runtime inputs you used in the notebook: session, sandbox, workspace, provider, and model.

```ts
export function createOpenClawCodingTools(options?: {
  agentId?: string;
  exec?: ExecToolDefaults & ProcessToolDefaults;
  messageProvider?: string;
  agentAccountId?: string;
  sandbox?: SandboxContext | null;
  sessionKey?: string;
  runSessionKey?: string;
  sessionId?: string;
  runId?: string;
  cwd?: string;
  workspaceDir?: string;
  config?: OpenClawConfig;
  modelProvider?: string;
  modelId?: string;
  modelApi?: string;
  // ...
});
```

Source: `src/agents/agent-tools.ts`

During attempt preparation, OpenClaw creates coding tools with the run's session, cwd, workspace, sandbox, provider, and model context:

```ts
const allTools = createOpenClawCodingTools({
  agentId: sessionAgentId,
  ...buildEmbeddedAttemptToolRunContext({ ...params, trace: runTrace }),
  exec: {
    ...params.execOverrides,
    config: params.config,
    elevated: params.bashElevated,
  },
  sandbox,
  sessionKey: sandboxSessionKey,
  sessionId: params.sessionId,
  runId: params.runId,
  cwd: effectiveCwd,
  workspaceDir: effectiveWorkspace,
  config: params.config,
  modelProvider: params.provider,
  modelId: params.modelId,
  // ...
});
```

Source: `src/agents/embedded-agent-runner/run/attempt.ts`

Tool policy is then applied in layers:

```ts
/** Applies configured policy layers to a tool list and emits deduped warnings/audit events. */
export function applyToolPolicyPipeline(params: {
  tools: AnyAgentTool[];
  toolMeta: (tool: AnyAgentTool) => { pluginId: string } | undefined;
  warn: (message: string) => void;
  steps: ToolPolicyPipelineStep[];
  auditLogLevel?: ToolPolicyAuditLogLevel;
}): AnyAgentTool[] {
  // ...
}
```

Source: `src/agents/tool-policy-pipeline.ts`

The visible `tool policy removed ...` message comes from audit code:

```ts
const matchedRuleSuffix = matchedRules.length > 0 ? `; matched ${matchedRules.join(", ")}` : "";
const message = `tool policy removed ${removed.length} tool(s) via ${rule}: ${toolNames.join(", ")}${matchedRuleSuffix}`;
const metadata = {
  rule,
  ruleKind,
  ...(matchedRules.length > 0
    ? {
        matchedRules,
        ...(truncated ? { matchedRulesTruncated: true } : {}),
      }
    : {}),
  removedToolCount: removed.length,
  removedTools: toolNames,
  removedToolsTruncated: truncated,
};
```

Source: `src/agents/tool-policy-audit.ts`

Tool use is not native model ability. OpenClaw registers tools, filters them through policy, runs allowed actions, and leaves audit evidence. This is a preview of Guardrails, not a full Guardrails lesson.


### 9. Session And Transcript

In Section 1, you used the same `--session-key` to give a temporary review preference and then review a sample answer. Here is the session history idea in plain language:

```text
Turn 1 in fundamentals-review-session:
  "When you review student answers, focus first on whether the answer cites observable OpenClaw behavior."

Turn 2 in the same session:
  "Review submissions/sample_student_answer.md using the review preference I just gave you."
```

The second turn can use the first turn because they share the same session key. A fresh session does not have that short-term history.

The session key resolves the persisted session bucket:

```ts
/**
 * Resolves the persisted session-store key for an inbound message.
 *
 * Explicit session keys pass through the compatibility normalizer, direct chats collapse to the
 * agent's canonical main bucket, and group/channel sessions stay isolated under the same agent.
 */
export function resolveSessionKey(
  scope: SessionScope,
  ctx: MsgContext,
  mainKey?: string,
  agentId: string = DEFAULT_AGENT_ID,
) {
  const explicit = ctx.SessionKey?.trim();
  if (explicit) {
    return normalizeExplicitSessionKey(explicit, ctx);
  }
  // ...
}
```

Source: `src/config/sessions/session-key.ts`

In plain language, `sessionKey` is the name you give to a task line. `sessionFile` is the saved file for that task line. A transcript is the saved record of messages, tool results, and runtime records for that session.

The session store and transcript files have separate responsibilities:

```ts
// Session store facade coordinates reads, writes, maintenance, delivery metadata, and exports.
```

Source: `src/config/sessions/store.ts`

```ts
// Session transcript facade resolves transcript files, appends mirror messages, and reads tails.
```

Source: `src/config/sessions/transcript.ts`

Inside the embedded attempt, OpenClaw opens the session file with a `SessionManager`:

```ts
await prewarmSessionFile(params.sessionFile);
const preparedUserTurnMessage = await params.userTurnTranscriptRecorder?.resolveMessage();
sessionManager = guardSessionManager(SessionManager.open(params.sessionFile), {
  agentId: sessionAgentId,
  sessionKey: params.sessionKey,
  config: params.config,
  // ...
});
```

Source: `src/agents/embedded-agent-runner/run/attempt.ts`

During a run, messages can be appended and the session context rebuilt:

```ts
activeSessionManager.appendMessage(
  redactedUserMessage as Parameters<typeof activeSessionManager.appendMessage>[0],
);
flushSessionManagerFile(activeSessionManager);
activeSession.agent.state.messages =
  activeSessionManager.buildSessionContext().messages;
```

Source: `src/agents/embedded-agent-runner/run/attempt.ts`

The `--session-key` is not just a display label. It affects the session file, session manager, transcript, and short-term task history. That is separate from `MEMORY.md`, which is a durable note in the workspace.


### 10. Memory Note And Memory Systems

OpenClaw memory starts with plain Markdown files in the agent workspace. The easiest one to see is `MEMORY.md`, which stores durable preferences and guidance.

In Section 1, you asked the agent to read `MEMORY.md`. That file was a visible memory note. You could open it yourself, ask the agent to use it, and compare the answer with the file content.

OpenClaw's memory overview describes three workspace memory files or file families:

- `MEMORY.md`: long-term memory for durable facts, preferences, and decisions.
- `memory/YYYY-MM-DD.md` or slugged variants: working notes for daily observations and session summaries.
- `DREAMS.md`: optional dreaming summaries for human review.

This lab only uses `MEMORY.md`. The `memory/*.md` notes can be indexed for tools such as memory search/get, but they are not the same as directly pasting every note into every prompt.

OpenClaw has a canonical root memory filename:

```ts
/** Canonical root memory file name used by current workspaces. */
export const CANONICAL_ROOT_MEMORY_FILENAME = "MEMORY.md";
/** Legacy root memory file name kept out of auxiliary scans. */
export const LEGACY_ROOT_MEMORY_FILENAME = "memory.md";

/** Resolves the canonical root memory file path for a workspace. */
export function resolveCanonicalRootMemoryPath(workspaceDir: string): string {
  return path.join(workspaceDir, CANONICAL_ROOT_MEMORY_FILENAME);
}
```

Source: `src/memory/root-memory-files.ts`

The system prompt gives `MEMORY.md` a specific role during an agent run:

```ts
lines.push(
  "MEMORY.md: durable user preferences and behavior guidance. Keep following it throughout the session unless higher-priority instructions override.",
);
```

Source: `src/agents/system-prompt.ts`

That visible file is not the whole OpenClaw memory system. Core config also names memory backend families for retrieval and session memory features:

```ts
/**
 * Memory config types shared by core context-engine paths and memory host/plugin runtimes.
 * Builtin memory stays core-owned; qmd settings describe the external QMD integration.
 */
// ...
/** Memory backend family selected for retrieval and session memory features. */
export type MemoryBackend = "builtin" | "qmd";
/** Citation rendering mode for memory-injected context. */
export type MemoryCitationsMode = "auto" | "on" | "off";
/** QMD search command flavor used for retrieval. */
export type MemoryQmdSearchMode = "query" | "search" | "vsearch";

/** Top-level memory config block. */
export type MemoryConfig = {
  backend?: MemoryBackend;
  citations?: MemoryCitationsMode;
  qmd?: MemoryQmdConfig;
};
```

Source: `src/config/types.memory.ts`

Read this in layers:

- `MEMORY.md` is the visible note in this lab. It is easy to inspect and easy to connect to the agent answer.
- `builtin` is the default memory backend. OpenClaw docs describe it as SQLite-based. It can index `MEMORY.md` and `memory/*.md` for keyword, vector, and hybrid search.
- `qmd` is an optional local-first search sidecar. It can add reranking, query expansion, extra indexed directories, and session transcript search. The docs call it QMD; they do not define a longer acronym, so this lab does not invent one.
- Honcho, LanceDB, `memory-core`, `active-memory`, and `memory-wiki` are memory-related systems or plugins. They are real parts of the OpenClaw memory ecosystem, but this lab only needs you to understand where `MEMORY.md` fits.

Source examples: `docs/concepts/memory.md`, `docs/concepts/memory-builtin.md`, `docs/concepts/memory-qmd.md`, `docs/concepts/memory-honcho.md`, `docs/plugins/memory-lancedb.md`, `extensions/memory-core`, `extensions/active-memory`, `extensions/memory-wiki`

For this lab, `MEMORY.md` is the easiest memory artifact to observe. It is not the entire OpenClaw memory system. Retrieval backends, indexing, promotion, wiki-style memory, and memory plugins are not covered in this lab.


### 11. The Agent Loop

After provider, context, tools, policy, session, and memory note are prepared, the run still needs a loop to execute the turn.

Read the loop order like this:

```text
new user task
  -> build current context
  -> call model provider
  -> model may request tools
  -> OpenClaw runs allowed tools
  -> tool results go back into context
  -> model continues or final answer returns
```

The lower-level loop starts with prompt messages and pushes lifecycle events:

```ts
/**
 * Start an agent loop with a new prompt message.
 * The prompt is added to the context and events are emitted for it.
 */
export function agentLoop(
  prompts: AgentMessage[],
  context: AgentContext,
  config: AgentLoopConfig,
  signal?: AbortSignal,
  streamFn?: StreamFn,
  runtime?: AgentCoreStreamRuntimeDeps,
): EventStream<AgentEvent, AgentMessage[]> {
  const stream = createAgentStream();

  void runAgentLoop(
    prompts,
    context,
    config,
    async (event) => {
      stream.push(event);
    },
    signal,
    streamFn,
    runtime,
  )
  // ...
}
```

Source: `packages/agent-core/src/agent-loop.ts`

In this signature, `prompts` are the new messages for this turn. `context` is the current model-facing state: system prompt, prior messages, and available tools. `streamFn` or runtime dependencies provide the provider stream, meaning the channel OpenClaw uses to receive model output from the selected provider.

Inside the loop, the assistant response can contain tool calls. When that happens, the loop executes the tools and pushes tool results back into the context as messages:

```ts
// Check for tool calls
const toolCalls = message.content.filter((c) => c.type === "toolCall");

const toolResults: ToolResultMessage[] = [];
hasMoreToolCalls = false;
if (toolCalls.length > 0) {
  const executedToolBatch = await executeToolCalls(
    currentContext,
    message,
    config,
    signal,
    emit,
  );
  toolResults.push(...executedToolBatch.messages);
  hasMoreToolCalls = !executedToolBatch.terminate;

  for (const result of toolResults) {
    currentContext.messages.push(result);
    newMessages.push(result);
  }
}
```

Source: `packages/agent-core/src/agent-loop.ts`

`toolResults` means the output returned by tools. The model does not secretly execute shell commands by itself. The loop receives a tool request, OpenClaw executes the allowed tool, and the tool result becomes another message the model can use.

The same file shows the model call boundary: messages are converted to LLM messages, tools are included in the LLM context, and the selected stream function calls the model provider.

```ts
// Convert to LLM-compatible messages (AgentMessage[] → Message[])
const llmMessages = await config.convertToLlm(messages);

// Build LLM context
const llmContext: Context = {
  systemPrompt: context.systemPrompt,
  messages: llmMessages,
  tools: context.tools,
};

const streamFunction = resolveAgentCoreStreamFn(runtime, streamFn);
```

Source: `packages/agent-core/src/agent-loop.ts`

The loop runs one turn by moving between model output, tool calls, tool results, and new context messages. The harness is larger than the loop because it also prepares the turn before the loop starts and records evidence after the loop runs.


### 12. What You Can Observe

Section 2 said that when an agent output is weak, you should inspect evidence instead of only blaming the model. OpenClaw leaves several kinds of evidence.

Some evidence is directly visible in the notebook output. Some evidence lives in session transcripts, diagnostics, or source-level lifecycle records.

| What you see | What it tells you | Where it comes from |
| --- | --- | --- |
| `openclaw models list` | The provider/model is registered and discoverable. | model registry and provider config |
| `openclaw infer model run` JSON | A direct model call can return text. | provider/model path |
| `[agents/tool-policy] ...` | OpenClaw filtered the available tools before the agent run. | tool policy audit |
| CLI/file inspection result | A tool action actually ran and returned output. | tool execution loop |
| Same-session vs fresh-session behavior | Session history changed what the agent knew. | session file and transcript |
| Final answer cites files or tools | The answer used workspace or tool evidence. | workspace/context/tool path |
| lifecycle phase/classification | OpenClaw recorded how the runtime attempt moved and ended. | lifecycle wrapper and diagnostics |

The lifecycle wrapper records phases around a harness attempt. A phase is a run stage, such as preparing, sending, or resolving. A classification is the final result label for the attempt.

```ts
/** Runs one harness attempt with diagnostics, tracing, and result classification. */
export async function runAgentHarnessLifecycleAttempt(
  harness: AgentHarness,
  params: AgentHarnessAttemptParams,
): Promise<AgentHarnessAttemptResult> {
  let result: AgentHarnessAttemptResult;
  let phase: AgentHarnessLifecyclePhase = "prepare";
  const startedAt = Date.now();
  // ...
  const runAndClassify = async () => {
    phase = "send";
    const rawResult = await harness.runAttempt(params);
    phase = "resolve";
    return applyAgentHarnessResultClassification(harness, rawResult, params);
  };
  // ...
}
```

Source: `src/agents/harness/lifecycle.ts`

Tool policy audit creates the visible tool-policy line:

```ts
const message = `tool policy removed ${removed.length} tool(s) via ${rule}: ${toolNames.join(", ")}${matchedRuleSuffix}`;
```

Source: `src/agents/tool-policy-audit.ts`

Session transcript code is responsible for transcript files and appends:

```ts
// Session transcript facade resolves transcript files, appends mirror messages, and reads tails.
```

Source: `src/config/sessions/transcript.ts`

Evidence is part of the harness. A useful agent system should leave enough records for you to debug the run. If the final report is weak, ask concrete questions: did it see the right files, were the needed tools available, did session history affect it, did the memory note matter, and does the answer cite evidence?


### 13. What To Remember

Keep these distinctions clear before leaving the lab:

- Provider is not runtime. DeepSeek, Ollama, vLLM, and SGLang are model-provider paths. They do not by themselves replace the OpenClaw agent runtime.
- Prompt is not the whole harness. Prompt assembly matters, but files, tools, action rules, session, memory notes, checks, and feedback loops also matter.
- Loop is not the whole harness. The loop runs one turn. The harness also prepares the turn and leaves evidence after it.
- `AgentHarness` is the source-level runtime contract. It is narrower than the course-level idea of harness engineering.
- A tool is not native model ability. OpenClaw registers tools, filters them, runs allowed actions, and returns results.
- Session is not long-term memory. Session is short-term task history. `MEMORY.md` is a durable workspace note.
- A stronger model does not automatically fix harness problems. Many problems require better files, clearer instructions, different tools, tighter action rules, tests, or evals.

Before leaving Section 3, check that you can answer these three questions about one run from Section 1:

1. Where did the provider/model enter the run?
2. Where did OpenClaw choose the runtime executor?
3. What evidence could you inspect if the final answer was weak?

That is the overall harness lesson: the model matters, but the system around the model decides what the agent can see, what it can do, what it remembers, and what you can inspect afterward.


### 14. References

OpenClaw documentation:

- Agent runtimes: https://docs.openclaw.ai/concepts/agent-runtimes
- Agent loop: https://docs.openclaw.ai/concepts/agent-loop
- Agent workspace: https://docs.openclaw.ai/concepts/agent-workspace
- System prompt: https://docs.openclaw.ai/concepts/system-prompt
- Memory overview: https://docs.openclaw.ai/concepts/memory
- Builtin memory engine: https://docs.openclaw.ai/concepts/memory-builtin
- QMD memory engine: https://docs.openclaw.ai/concepts/memory-qmd
- Agent CLI: https://docs.openclaw.ai/cli/agent
- Infer CLI: https://docs.openclaw.ai/cli/infer
- Models CLI: https://docs.openclaw.ai/cli/models
- DeepSeek provider: https://docs.openclaw.ai/providers/deepseek
- Ollama provider: https://docs.openclaw.ai/providers/ollama

Harness engineering background:

- OpenAI, Harness Engineering: https://openai.com/index/harness-engineering/
- Martin Fowler, Harness Engineering: https://martinfowler.com/articles/harness-engineering.html
- Chinese secondary interpretation: https://zhuanlan.zhihu.com/p/2014014859164026634
